# 01 — BCRD Exchange Rates | Automatic Download (raw landing)

Downloads the official BCRD daily USD/DOP reference-rate workbook and lands it, unchanged, in the raw archive
`/Volumes/rd_economic_intelligence/bronze/bcrd_raw/exchange_rates/`.

| Step | Rule |
|---|---|
| Source | `https://cdn.bancentral.gov.do/documents/estadisticas/mercado-cambiario/documents/TASA_DOLAR_REFERENCIA_MC.xlsx` — stable URL, daily buy/sell rates **since January 1991** (sheet `Diaria`; other sheets hold monthly, quarterly and annual averages). It replaces the manual 5-year "Histórico diario de tasas" export, which has no stable URL. |
| Validate | Sheet `Diaria` with the header `Año, Mes, Día, Compra, Venta` in its first 10 rows; otherwise the task fails and nothing is written. |
| Land | Shared helper `src/shared/acquisition.py`: identical content is never landed twice; new content goes to `<yyyy>/<mm>/`; a republished file keeps its own version (`<name>__<date>`). |
| Source down | If the site cannot be reached after 3 attempts (network, invalid TLS certificate, HTTP 5xx), the task ends as a warning (`SOURCE_UNAVAILABLE`), the outage is logged in `quality.source_availability` and the source keeps its last data. HTTP 4xx and layout changes still fail. |
| Signal | Task value `downloaded`. |

**Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, io, json, importlib, itertools
import openpyxl

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
importlib.reload(ACQ)

LANDING = "/Volumes/rd_economic_intelligence/bronze/bcrd_raw/exchange_rates/"
URL = "https://cdn.bancentral.gov.do/documents/estadisticas/mercado-cambiario/documents/TASA_DOLAR_REFERENCIA_MC.xlsx"
HEADER = ["Año", "Mes", "Día", "Compra", "Venta"]

try:
    data = ACQ.http_get(URL)
    wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True, data_only=True)
    if "Diaria" not in wb.sheetnames:
        raise ValueError(f"Structural contract failed: sheet 'Diaria' not found ({wb.sheetnames})")
    top = [[str(c).strip() if c is not None else None for c in r[:5]] for r in itertools.islice(wb["Diaria"].iter_rows(values_only=True), 0, 10)]
    if HEADER not in top:
        raise ValueError(f"Structural contract failed: header {HEADER} not found in the first 10 rows of 'Diaria'")

    result = ACQ.land(data, URL.rsplit("/", 1)[-1], LANDING)
    dbutils.jobs.taskValues.set(key="downloaded", value=result["status"] == "DOWNLOADED")
    dbutils.notebook.exit(json.dumps(result, ensure_ascii=False))
except ACQ.SourceUnavailable as e:
    ACQ.source_unavailable(dbutils, spark, "bcrd_exchange_rates", e)